In [ ]:
import sys
from setup_params import params
from setup_paths import paths
sys.path.append(paths["project_dir"])

In [ ]:
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import shapely as shp

from copy import deepcopy
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from os.path import join
from pathlib import Path
from src.prep_and_plot import prep_city, prepare_downloaded_map, gen_noisy_demand, gen_noisy_penalty, create_demand, hex_split_pop_density
from src.prep_and_plot.data_helper import get_polygon_from_json, remove_dead_end_edges
from src.prep_and_plot.plot import plot_graph

In [ ]:
city_name = "Dresden"
save = "dd"

output_folder = join(paths["input_folder"], save)
Path(output_folder).mkdir(parents=True, exist_ok=True)
Path(join(paths["plot_folder"], save)).mkdir(parents=True, exist_ok=True)

german_pop_dist_file = join(paths["data_dir"], "demand", "kontur_population_DE_20231101.gpkg")
local_pop_dist_file = join(paths["data_dir"], "demand", f"{save}_pop_inner_split.gpkg")

polygon_file = join(paths["data_dir"], "polygons", f"{save}.json")
elevation_raster_file = join(paths["data_dir"], "elevation", "DGM1_Dresden.tif")

buffered_graph_file = join(paths["input_folder"], save, f"{save}_buffered.graphml")
graph_file = join(paths["input_folder"], save, f"{save}_base.graphml")

input_csv = join(paths["data_dir"], "csvs", f"{save}.csv")

global_crs = 4326
local_crs = 32633

intersection_tol = 20
dead_end_tol = 100

len_scale_all = 10000
len_scale_bike = 2000   # -> Maximum of f(L) at 2*len_scale_bike
modal_share_bike = 0.2

node_dist_cutoff = 400

buffer_dist = 5000
nw_type = "drive"

cmap_demand = plt.get_cmap("viridis")
plot_params = {"plot_format": "svg", "dpi": 300, "figsize_map": (3, 4), "fs_title": 10, "fs_cbar_label": 6, "fs_cbar_ticks": 6, "edge_lw_thick": 0.75, "edge_lw_thin": 0.25, "edge_color": "#808080", "node_size": 0.5}

In [ ]:
polygon = get_polygon_from_json(polygon_file)

polygon_proj = gpd.GeoDataFrame({"geometry": [polygon]}, geometry="geometry", crs=global_crs)
polygon_proj = polygon_proj.to_crs(local_crs)["geometry"][0]

poly_buffered_proj = shp.buffer(polygon_proj, buffer_dist)
poly_buffered = gpd.GeoDataFrame({"geometry": [poly_buffered_proj]}, geometry="geometry", crs=local_crs)
poly_buffered = poly_buffered.to_crs(global_crs)["geometry"][0]

In [ ]:
useful_tags = (
        ox.settings.useful_tags_way
        + ["cycleway"]
        + ["cycleway:right"]
        + ["cycleway:left"]
        + ["bicycle"]
        + ["sidewalk:both:bicycle"]
        + ["sidewalk:right:bicycle"]
        + ["sidewalk:left:bicycle"]
        + ["cycleway:both:bicycle"]
        + ["cycleway:right:bicycle"]
        + ["cycleway:left:bicycle"]
    )
ox.settings.useful_tags_way = useful_tags
ox.settings.bidirectional_network_types = ox.settings.bidirectional_network_types + [nw_type]

In [ ]:
g_base = ox.graph_from_polygon(poly_buffered, network_type=nw_type, simplify=False, retain_all=True)
g_bike = ox.graph_from_polygon(poly_buffered, network_type="bike", simplify=False, retain_all=True)
bbox = ox.convert.graph_to_gdfs(g_base, nodes=False).total_bounds

In [ ]:
osmids_elbe = [
    802134559, 256886205, 274223567, 758359329, 337015190, 337015192, 53181046, 25749410, 24368030, 337015191, 1063423086, 60611096, 128777057, 128777056, 256886199, 332178495, 68424380, 260443503, 245623208, 260443518, 1417773298, 1417773297, 260443506, 260449306, 260449301, 1061866939, 260449298, 111989498, 260449304, 389080884, 389080887, 260449305, 260449299, 260449300, 25687960, 260449297, 260455943, 389080885, 389080886, 260455921, 14038596, 172543676, 172543677, 114647648, 30117723, 260455933, 106976620, 30117730, 25924360, 260455935, 181266172, 173457564, 173457566, 173457565, 173457563, 260455931, 260455956, 79857954, 79857959, 988417923, 4690968, 988417924, 159064861, 4927015, 4927016, 963766411, 963893726, 707636703, 1284498138, 707636702, 23162489, 4927137, 260455949, 260455938, 260455916, 159064847, 33708558, 159064846, 126821712, 126821714, 126821707, 30183875, 30183876, 126821717, 126821709, 126821703, 126821704, 126821705, 4690969, 126821741, 293399770, 126821745, 683428628, 25525493, 126821805, 11912940, 159064866, 1042579376, 322202471, 963893728, 1063925068, 175725521, 963893727, 746351027, 766651946, 37984294, 4499914, 4499913, 766748804, 4211902, 766748803, 766748801, 766748802, 766748800, 766726237, 766726236, 1155213982, 361599808, 239490404, 361599776, 239490406, 361599827, 239490405, 766726245, 239490415, 239490408, 860435317, 239490418, 766726231, 766726229, 361599820, 239490411, 370748259, 239490420, 860435316, 860435315, 345548988, 53726386, 325582371, 325582374, 325582367, 25546241, 325582378, 860435314, 25091543, 241700149, 175725446, 87538747, 173166646, 684124951, 126821721, 298723026, 172614779, 683428637, 34023233, 540749721, 76984816, 314293855, 1420793436, 343484852, 1020091464, 1384430697, 913600676, 1384430695, 1020091466, 241948041, 24925546, 684025669, 175759337, 684025670, 1044808455, 684023332, 1315323746, 24925545, 1392233650, 9043016, 1044808454, 165930648, 770397352, 770397351, 233521436, 325017789, 102319893, 987315162, 102319894, 27202073, 9821582, 678119066, 165930647, 102319890, 26818860, 1044808453, 683381345, 1044808451, 25399065, 1044808450, 125674037, 125674045, 1392233654, 683381368, 125674018, 674250615, 1392233657, 770892090, 1108602410, 125674019, 4604933, 233090039, 254555712, 254555706, 980374167, 254555707, 980374169, 28506354, 254555711, 980374168, 254555713, 214606405, 1378178441, 1378178442, 28954964, 28485022, 1218712572, 1218712571, 203820333, 1218712570, 1218712569, 771310802, 923211533, 288961137, 970369971, 203820331, 163426412, 220070733, 447081824, 220070736, 447081823, 233783715, 447081820, 346520724, 28485058, 30131232, 1207209009, 9875691, 577557442, 219663929, 351206693, 447081807, 683395509, 1305608020, 233781250, 233781252, 1208287878, 233781251, 28605171, 233783128, 683395517, 30131257, 30899791, 5057375, 9876039, 646924652, 26790667, 397803129, 30598605, 397803128, 9097639, 27202072, 770397339, 770397340, 165930645, 9097586, 9711993, 683381346, 683381350, 125674025, 121263917, 121263925, 25478377, 24823680, 332078362, 265273346, 265273340, 265273359, 186144781,
    849823517, 849823515, 27694726, 150289827, 218715257, 418465949, 218715258, 218712947, 218712945, 218712946, 232347747, 985860642, 232347749, 351673904, 351673909, 42375311, 22816781, 85565514, 114392939, 85565515, 33804552, 41548243, 741650925, 741650924, 41548244, 114392955, 114392970, 207102373, 207102374, 41046737, 1390136937, 22101057, 114392942, 114392915, 114392919, 114392922, 114392967, 114392934, 114392971, 114392976, 114392969, 114392913, 22102314, 980928197, 980928198, 22102552, 324646324, 114392964, 23359423, 22103295, 706327233, 22090640, 22090497, 76448192, 114392959, 168462348, 173457593, 168462349, 126821806, 4927138, 339251140, 106937676, 106937607, 4236428, 4236429, 4236427, 23162502, 106989525, 106937723, 994054237, 106937642, 114392981, 114392961, 51381274, 4499921, 4499920, 4499919, 4398978, 994054238, 586383044, 586383045, 994054239, 24930138, 126821716, 126821706, 24930170, 4393640, 4393645, 760150051, 4390738, 1239588452, 246703498, 760150054, 760150055, 427251031, 4287882, 124859633, 1237876372, 996102659, 759408332, 1000331062, 4604934, 50443882, 1000331061, 142844987, 105808048, 1000331060, 242302225, 250929455, 759425299, 250929454, 770029593, 249883408, 1317388894, 255598300, 388182402, 988229767, 770029597, 241234028, 1315323745, 250201135, 1317388895, 1314868780, 1330165219, 1039014665, 1058389081, 1058389082, 4604938, 1039014664, 113599086, 1039014663, 125674015, 56428972, 70927246, 70926957, 4604939, 70926176, 186780206, 56428976, 1180726921, 249954300, 1180726922, 56428975, 56428974, 186780207, 181266178, 854333251, 24641140, 24641138, 113599078, 24641137, 24641136, 4489269, 51795125, 51795131, 51795129, 51795128, 113599065, 26790602, 35318255, 216737714, 4773675, 113599091, 10300205, 981326296, 113599072, 978227825, 113599083, 22335727, 1302134756, 863533924, 113599069, 863533925, 28591169, 368365634, 28591171, 28591170, 1061189280, 545112134, 28121859, 23567599, 4516854, 311161423, 311161424, 25398718, 1384430696, 1346644228, 1346661279, 4290583, 25398617, 210599930, 273909802, 377710054
]
osmids_heide = [
    925831677, 236076614, 236369234, 236369233, 170810950, 236430996, 400529140, 400529139, 28196408, 103913128, 370870656, 400616578, 370870657, 1329159896, 684106658, 684106659, 667642369, 684106660, 307544931, 9706322, 1112487543, 1112487557, 23764051, 4720282, 4383496, 1057824541, 138654556, 1057824540, 28098589, 1202109167, 82222388, 1202109388, 1202109389, 1202108680, 818015495, 293240027, 366206612, 27115278, 27115279, 170810956, 293243317, 366131301, 818015494, 366131302, 292624655, 817875960, 32149782, 293240028, 366206607, 292624657, 27115280, 26980498, 1202105153, 636621966, 1202105324, 1202105154, 1183639488, 636621965, 27940966, 28408645, 23121143, 636621963, 636621964, 1099681945, 27940860, 23121145, 131003564, 21896812, 165930714, 53546011, 53546009, 636574930, 131711859, 1092128866, 21898986, 21898989, 21898991, 4699421, 31265128, 4522413, 53177225, 1091962047, 1091962048, 4522414, 26291113, 484347595, 26291112, 37178310, 1095474630, 4522417, 23012337, 51515743, 13753777, 79193322, 121263927, 165930840, 46985611, 137702209, 137702210, 24587156, 1089956340, 24587157, 1089956341, 369242612, 4821468, 1304824676, 170812846, 4940664, 233951620, 43217481, 1304713148, 143258354, 1304713149, 1304713150, 1304713147, 1304713153, 1304713152, 1304713151, 170813829, 10210697, 478848310, 23898271, 80184772, 1172039254, 1172039253, 1172039252, 1172039251, 1172039250, 78445035, 561325821, 1300995568, 561325824, 1300995569, 23199791, 204179404, 13770025, 4940765, 804857396, 33096232, 23613192, 23613193, 1173093027, 28054951, 28054994, 44735867, 955582131, 33898059, 38591157, 1283896181, 1283896182, 1283896183, 1283896184, 1283896185, 1283896186, 1283896187, 1283896188, 43922019, 1283896189, 1066881109, 1412483325, 1412483324, 183903616, 71222659, 719033843, 33096242, 24354626, 423112569, 78555472, 52722137, 27946536, 824551788, 179105505, 179105898, 50552225, 37804912, 477691670, 37832973, 24001414, 1273940936, 32892136, 965570249, 32892057, 33901094, 33901096, 57729198, 83412921, 1368777419, 24556343, 50552223, 50033122, 33901036, 552863308, 1412471633, 33901037, 175759333, 30134386, 33901022, 664865849, 31153823, 553042563, 553042562, 24556841, 24556842, 33901018, 24556845, 980695272, 980695271, 24556846, 53396609, 24558457, 35173136, 1412471635, 98490613, 175759324, 29338397, 469993882, 319884236, 53390469, 1352104191, 1352104189, 29338396, 1352104190, 145815300, 145815299, 1260709837, 882961730, 60822009, 60408719, 1340711722, 23245526, 965560817, 85933723, 145813186, 85814872, 175759332, 175759329, 175759326, 965562787, 175759338, 27027090, 27027089, 1283415246, 965562785, 1283415245, 1283415244, 144931246, 136114183, 25353100, 422657523, 1208841995, 1208841999, 23986840, 809481416, 809481417, 915085072, 1207209148, 25856698, 23574940, 683395549, 800298238, 800298237, 23574941, 683395547, 23325376, 30598655, 397803128, 30598605, 397803129, 26790667, 646924652, 9876039, 23597025, 237300021, 23597026, 66059287, 34762158, 34762155, 497205660, 66053794, 497205661, 23362365, 683395514, 25043198, 66058459, 66058385, 175759327, 91664948, 27809888, 427674772, 577557441, 427674771, 28485080, 23798868, 854916565, 57821688, 23798858, 23798853, 28485058, 346520724, 447081820, 233783715, 447081823, 220070736, 447081824, 220070733, 163426412, 203820331, 970369971, 203820332, 163426411, 220070734, 220070994, 25478377, 1218712571, 1218712572, 28485022, 28954964, 1378178442, 1378178441, 214606405, 254555713, 980374168, 254555711, 28506354, 980375054, 4731093, 27102879, 25877747, 9794888, 30130504, 1183380259, 740164865, 1183380256, 25399240, 1083888071, 238974922, 4604933, 125674019, 1108602410, 770892090, 1392233657, 674250615, 125674018, 683381368, 1392233654, 125674045, 125674037, 1044808450, 25399065, 1044808451, 683381345, 1044808453, 26818860, 102319890, 165930647, 678119066, 9821582, 27202073, 102319894, 987315162, 102319893, 325017789, 233521436, 770397351, 770397352, 165930648, 1044808454, 1392233650, 9043016, 24925545, 1315323746, 684023332, 1044808455, 684025670, 60611099, 60611110, 458245614, 169952324, 1316788577, 29160559, 211531241, 169952373, 125418111, 125418108, 23166722, 23166721, 4313322, 37217417, 4313323, 1091958960, 714890665, 4635478, 1009338107, 25153204
]
osmids_gg = [1317687949, 1317687950]

In [ ]:
bp_elbe = [(u, v, k) for u, v, k, e_osmid in g_bike.edges(keys=True, data="osmid") if e_osmid in osmids_elbe]
g_bp_elbe = g_bike.edge_subgraph(bp_elbe)
nx.set_edge_attributes(g_bp_elbe, values={(u, v, k): "service" for u, v, k, d in g_bp_elbe.edges(keys=True, data="highway") if d == "unclassified"}, name="highway")
bp_heide = [(u, v, k) for u, v, k, e_osmid in g_bike.edges(keys=True, data="osmid") if e_osmid in osmids_heide]
g_bp_heide = g_bike.edge_subgraph(bp_heide)
nx.set_edge_attributes(g_bp_heide, values={(u, v, k): "service" for u, v, k, d in g_bp_heide.edges(keys=True, data="highway") if d == "unclassified"}, name="highway")
g_bp_gg = ox.graph_from_place("Dresden", network_type=nw_type, custom_filter=['["name"~"Hauptallee"]','["name"~"Querallee"]','["name"~"Fürstenallee"]'], simplify=False, retain_all=False)
g_bp_gg = nx.compose_all([g_bp_gg, g_bike.edge_subgraph( [(u, v, k) for u, v, k, e_osmid in g_bike.edges(keys=True, data="osmid") if e_osmid in osmids_gg])])
nx.set_edge_attributes(g_bp_gg, values="service", name="highway")

In [ ]:
g_buffered = nx.compose_all([g_base, g_bp_elbe, g_bp_heide, g_bp_gg])
g_buffered = prepare_downloaded_map(g_buffered, ensure_highway_diff=True, consolidate=True, intersection_tol=intersection_tol, remove_dead_ends=True, dead_end_tol=dead_end_tol, elevation_raster_file=elevation_raster_file)
ox.save_graphml(g_buffered, buffered_graph_file)

In [ ]:
g = ox.truncate.truncate_graph_polygon(g_buffered, polygon)
g = ox.truncate.largest_component(g, strongly=True)
g = remove_dead_end_edges(g, dead_end_tol=dead_end_tol)
print(f"Observed area graph with {len(g.nodes())} nodes and {len(g.edges())} edges.")
ox.save_graphml(g, graph_file)

In [ ]:
edge_grade = [d for _, _, d in g.edges(data="grade")]
edge_color = list()
edge_zorder = list()
for d in edge_grade:
    if d > 0.06:
        edge_color.append("r")
        edge_zorder.append(4)
    elif d > 0.04:
        edge_color.append("C1")
        edge_zorder.append(3)
    elif d > 0.02:
        edge_color.append("y")
        edge_zorder.append(2)
    else:
        edge_color.append("#808080")
        edge_zorder.append(1)
edge_zorder = [d for _, _, d in g.edges(data="grade")]
plot_graph(g, figsize=(20, 20), node_size=0, edge_color=edge_color, edge_zorder=edge_zorder, edge_linewidth=0.5)

In [ ]:
edge_grade = [d for _, _, d in g.edges(data="grade")]
edge_color = list()
edge_zorder = list()
for d in edge_grade:
    if d > 0.06:
        edge_color.append("r")
        edge_zorder.append(4)
    elif d > 0.04:
        edge_color.append("C1")
        edge_zorder.append(3)
    elif d > 0.02:
        edge_color.append("y")
        edge_zorder.append(2)
    else:
        edge_color.append("#808080")
        edge_zorder.append(1)
edge_zorder = [d for _, _, d in g.edges(data="grade")]
plot_graph(g, figsize=(20, 20), node_size=0, edge_color=edge_color, edge_zorder=edge_zorder, edge_linewidth=0.5)

In [ ]:
total_nw_len = sum(d for u, v, d in g.edges(data="length")) / 1000
total_ex_inf_len = sum(d['length'] for u, v, d in g.edges(data=True) if d['ex_inf'] or d['highway'] == 'bike_path')  / 1000
print(f"Total network length: {total_nw_len:.0f} km")
print(f"Existing infrastructure length: {total_ex_inf_len:.0f} km, relative {total_ex_inf_len/total_nw_len:.3f}")

In [ ]:
plot_graph(g, node_size=0, edge_linewidth=0.5)

In [ ]:
bottom, top, left, right = 51.105437917278664, 51.111789848754796, 13.827002297969557, 13.838352064917416
zoom_bbox = [left, bottom, right, top]
st_colors = {"bike_path": "C0", "residential": "C1", "tertiary": "C2", "secondary": "C3", "primary": "C4"}
plot_graph(g, node_size=0, edge_linewidth=0.5, edge_color=[st_colors[d] for _,_, d in g.edges(data="highway")]) #, bbox=zoom_bbox

# Demand generation

In [ ]:
german_pop_dist = gpd.read_file(german_pop_dist_file)
german_pop_dist.to_crs(local_crs, inplace=True)
german_pop_dist["centroid"] = german_pop_dist.centroid

german_pop_dist["in_buffered_area"] = poly_buffered_proj.contains(german_pop_dist.centroid)

buffered_local_data = german_pop_dist.loc[german_pop_dist["in_buffered_area"], ["h3", "population", "geometry", "centroid"]]
buffered_local_data.reset_index(drop=True, inplace=True)
buffered_local_data.rename_axis("station", inplace=True)
buffered_local_data.reset_index(inplace=True)

buffered_local_data["inside_study_area"] = polygon_proj.contains(buffered_local_data.centroid)
buffered_local_data["outside_study_area"] = ~polygon_proj.contains(buffered_local_data.centroid)

inside_data = buffered_local_data.loc[buffered_local_data["inside_study_area"], ["h3", "population", "geometry", "centroid"]]
inside_data.reset_index(drop=True, inplace=True)
outside_data = buffered_local_data.loc[buffered_local_data["outside_study_area"], ["h3", "population", "geometry", "centroid"]]
inside_data_split = hex_split_pop_density(inside_data, local_crs, global_crs, split_size=3).reset_index(drop=True).to_crs(local_crs)
inside_data_split["centroid"] = inside_data_split.centroid
inside_data_split.reset_index(drop=True, inplace=True)

local_pop_dist = pd.concat([inside_data_split, outside_data])
local_pop_dist.reset_index(drop=True, inplace=True)
local_pop_dist.drop(columns=["centroid"], inplace=True)
local_pop_dist.to_crs(global_crs, inplace=True)
local_pop_dist.to_file(local_pop_dist_file, driver="GPKG")

In [ ]:
demand, _, _, od_matrix, od_distances, od_paths, stations, node_mapping, length_scale_general, length_scale_bike, modal_share, p_bike = create_demand(
    pop_dist_file=local_pop_dist_file,
    polygon_file=polygon_file,
    graph_file=buffered_graph_file,
    modal_share=modal_share_bike,
    length_scale_general=len_scale_all,
    length_scale_bike=len_scale_bike,
    buffer_dist=buffer_dist,
    crs_glob=global_crs,
    crs_loc=local_crs,
    node_dist_cutoff=node_dist_cutoff,
    save_name=save,
    paths=paths,
)

In [ ]:
local_pop_dist = gpd.read_file(local_pop_dist_file)
local_pop_dist.to_crs(local_crs, inplace=True)
local_pop_dist["centroid"] = local_pop_dist.centroid

g_inside = ox.truncate.truncate_graph_polygon(g_buffered, polygon)
g_outside = g_buffered.copy()
edges_inside = [e for e in g_buffered.edges() if e in g_inside.edges()]
g_outside.remove_edges_from(edges_inside)

gdf_buffered = ox.graph_to_gdfs(g_buffered, nodes=False).to_crs(local_crs)
gdf_inside = ox.graph_to_gdfs(g_inside, nodes=False).to_crs(local_crs)
gdf_outside = ox.graph_to_gdfs(g_outside, nodes=False).to_crs(local_crs)

stations["geometry_boundary"] = stations.set_geometry("geometry").boundary
stations["area"] = stations.set_geometry("geometry").area
stations["pop_density"] = stations["population"] / stations["area"]

In [ ]:
fig_g_buffered, ax_g_buffered = plt.subplots(dpi=plot_params["dpi"], figsize=plot_params["figsize_map"])

plot_graph(
    g_buffered,
    ax=ax_g_buffered,
    node_size=0,
    edge_color=["k" if e in edges_inside else "#808080" for e in g_buffered.edges()],
    edge_linewidth=plot_params["edge_lw_thin"],
)

fig_g_buffered.savefig(join(paths["plot_folder"], save, f"{save}_buffered_graph.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

In [ ]:
fig_pop, ax_pop = plt.subplots(dpi=plot_params["dpi"], figsize=plot_params["figsize_map"])

population_var = "population"
gdf_inside.plot(ax=ax_pop, color="k", lw=plot_params["edge_lw_thin"])
gdf_outside.plot(ax=ax_pop, color="#808080", lw=plot_params["edge_lw_thin"])

stations.set_geometry("geometry_boundary").plot(ax=ax_pop, column=population_var, cmap=cmap_demand, lw=plot_params["edge_lw_thin"], zorder=10, alpha=0.5)
stations.set_geometry("location").plot(ax=ax_pop, column=population_var, cmap=cmap_demand, markersize=plot_params["node_size"], zorder=10)

sm = plt.cm.ScalarMappable(
    cmap=cmap_demand,
    norm=Normalize(vmin=stations[population_var].min(), vmax=stations[population_var].max())
)
divider = make_axes_locatable(ax_pop)
cbaxes = divider.append_axes("bottom", size="5%", pad=0.05)
cbar = fig_pop.colorbar(
    sm,
    ax=ax_pop,
    orientation="horizontal",
    cax=cbaxes,
)
cbar.set_label(label="population", size=plot_params["fs_cbar_label"])
cbar.ax.tick_params(labelsize=plot_params["fs_cbar_ticks"])
ax_pop.set_axis_off()

fig_pop.savefig(join(paths["plot_folder"], save, f"{save}_pop_dens.{plot_params['plot_format']}"), bbox_inches="tight")
plt.show()

# Data preparation

In [ ]:
prep_city(
    city_name,
    save,
    input_csv,
    polygon_file=polygon_file,
    cached_graph=True,
    cached_graph_file_path=graph_file,
    paths=paths,
    params=params,
)

In [ ]:
rng = np.random.default_rng(seed=12345)

g_algo = ox.load_graphml(join(paths["input_folder"], save, f"{save}.graphml"))
edge_list = [(u, v) for u, v, d in g_algo.edges(data="ramp") if not d]
for i in range(1, 11):
    random_order_i = deepcopy(edge_list)
    rng.shuffle(random_order_i)
    with open(join(paths["input_folder"], save, f"{save}_random_order_{i}.json"), "w") as fp:
        json.dump(random_order_i, fp)

# Noisy Data

In [ ]:
rng = np.random.default_rng(seed=12345)

In [ ]:
with open(join(paths["input_folder"], save, f"{save}_demand.json"), "r") as f:
    demand = json.load(f)

In [ ]:
car_penalty = {"primary": [1/7], "secondary": [1/2.4], "tertiary": [1/1.4], "residential": [1/1.1], "bike_path": [1], "superhighway": [1]}
slope_penalty = {0.06: [1/3.2], 0.04: [1/2.2], 0.02: [1/1.4], 0.0: [1/1.0]}
intersection_penalty = {"large": [30], "medium": [15], "small": [0]}
turn_penalty = {"large": [15], "medium": [7.5], "small": [0]}

penalties_dict = {
    "car_penalty": car_penalty,
    "slope_penalty": slope_penalty,
    "intersection_penalty": intersection_penalty,
    "turn_penalty": turn_penalty,
}

with open(join(paths["input_folder"], save, f"{save}_penalties.json"), "w") as fp:
    json.dump(penalties_dict, fp)

max_penalties = {"car_penalty": 1, "slope_penalty": 1, "intersection_penalty": np.inf, "turn_penalty": np.inf, "edge_time_penalty": 0, "bp_end_penalty": np.inf}
min_penalties = {"car_penalty": 0, "slope_penalty": 0, "intersection_penalty": 0, "turn_penalty": 0, "edge_time_penalty": 0, "bp_end_penalty": 0}

In [ ]:
delta_noise = 0.20
for i in range(1, 11):
    print(f"Generating data set {i}")
    gen_noisy_demand(demand, delta_noise, save, f"{save}_noisy_demand_{i}", rng=rng)
    with open(join(paths["input_folder"], f"{save}_noisy_demand_{i}", f"{save}_noisy_demand_{i}_penalties.json"), "w") as fp:
        json.dump(penalties_dict, fp)
    for pen, pen_dict in penalties_dict.items():
        min_penalty = min_penalties[pen]
        max_penalty = max_penalties[pen]
        gen_noisy_penalty(penalties_dict, pen, delta_noise, save, f"{save}_noisy_{pen}_{i}", min_penalty=min_penalty, max_penalty=max_penalty, rng=rng)